# Full FP16 XE epoch and VALID5000 comparison
Select GPU T4 x2, enable Internet and attach the four inputs listed below. Save & Run All. TRAIN uses mixed precision. VALID5000 beam search and NLL use FP32. The run starts fresh and has a three-hour cap. Send the review ZIP after it finishes.


In [ ]:
import hashlib,json,os,queue,signal,subprocess,sys,tempfile,threading,time,traceback,urllib.request,zipfile,shutil,tarfile
from pathlib import Path
REVISION='5eb012279e6fd0b5e47a56710ed5e3efa7449d2f'
MAPPING={'experiments/clip_patch_baseline/full_data.py': 'full_data.py', 'experiments/clip_patch_baseline/full_runtime.py': 'full_runtime.py', 'experiments/clip_patch_baseline/run_full.py': 'run_full.py', 'experiments/clip_patch_baseline/patch_model.py': 'patch_model.py', 'experiments/clip_patch_baseline/patch_beam.py': 'patch_beam.py', 'experiments/clip_patch_baseline/score_full.py': 'score_full.py', 'experiments/clip_patch_baseline/patch_data.py': 'patch_data.py', 'experiments/clip_patch_baseline/test_full_data.py': 'test_full_data.py', 'experiments/clip_patch_baseline/make_full_notebook.py': 'make_full_notebook.py', 'experiments/clip_patch_baseline/SMOKE_SOURCE.json': 'SMOKE_SOURCE.json', 'experiments/clip_patch_baseline/FULL_XE10_PROTOCOL.md': 'FULL_XE10_PROTOCOL.md', 'experiments/csa_paper/data.py': 'data.py', 'experiments/csa_paper/real_bank_data.py': 'real_bank_data.py', 'experiments/csa_paper/cuda_runtime_guard.py': 'cuda_runtime_guard.py', 'experiments/csa_paper/saved_quality_data.py': 'saved_quality_data.py', 'experiments/clip_patch_baseline/local_stage.py': 'local_stage.py', 'experiments/clip_patch_baseline/test_local_stage.py': 'test_local_stage.py', 'experiments/clip_patch_baseline/LOCAL_STAGE_V3_PROTOCOL.md': 'LOCAL_STAGE_V3_PROTOCOL.md', 'experiments/clip_patch_baseline/semantic_data.py': 'semantic_data.py', 'experiments/clip_patch_baseline/semantic_patch_model.py': 'semantic_patch_model.py', 'experiments/clip_patch_baseline/semantic_training.py': 'semantic_training.py', 'experiments/clip_patch_baseline/test_semantic_data.py': 'test_semantic_data.py', 'experiments/clip_patch_baseline/BANK_PROMPT_SCRATCH_PROTOCOL.md': 'BANK_PROMPT_SCRATCH_PROTOCOL.md', 'experiments/csa_paper/model.py': 'bank_layers.py', 'experiments/csa_paper/SAVE_BANK_V130_PACKAGE.json': 'SAVE_BANK_V130_PACKAGE.json', 'experiments/csa_paper/lemma_correction_review_text/bank_words_corrected.json': 'bank_words_corrected.json', 'experiments/csa_paper/FULL_PROMPT_FRESH_V175_RESULT.json': 'FULL_PROMPT_FRESH_V175_RESULT.json', 'experiments/clip_patch_baseline/memory_fusion.py': 'memory_fusion.py', 'experiments/clip_patch_baseline/csa_full_data.py': 'csa_full_data.py', 'experiments/clip_patch_baseline/csa_full_runtime.py': 'csa_full_runtime.py', 'experiments/clip_patch_baseline/run_csa_full.py': 'run_csa_full.py', 'experiments/clip_patch_baseline/score_csa_full.py': 'score_csa_full.py', 'experiments/clip_patch_baseline/test_csa_full_data.py': 'test_csa_full_data.py', 'experiments/clip_patch_baseline/csa_patch_model.py': 'csa_patch_model.py', 'experiments/clip_patch_baseline/csa_sharing_data.py': 'csa_sharing_data.py', 'experiments/clip_patch_baseline/gradient_agreement.py': 'gradient_agreement.py', 'experiments/clip_patch_baseline/test_gradient_agreement.py': 'test_gradient_agreement.py', 'experiments/clip_patch_baseline/CSA_SHARED_FULL_XE10_PROTOCOL.md': 'CSA_SHARED_FULL_XE10_PROTOCOL.md', 'experiments/clip_patch_baseline/make_csa_full_notebook.py': 'make_csa_full_notebook.py'}
HASHES={'experiments/clip_patch_baseline/full_data.py': 'ab69cf61d4ad0456d9a869929750ab10f3a2a33b6c76683689539e57e4015a74', 'experiments/clip_patch_baseline/full_runtime.py': '230f7e10d96005fbffc33fb151a7a1d7c7245326e73826134a400772dadd04bb', 'experiments/clip_patch_baseline/run_full.py': '32dfb92228fad22d4f503680db1c8226db1e6abf9d7dd253786c3f16be9a97c0', 'experiments/clip_patch_baseline/patch_model.py': 'dfe6b1d83bfc1e3c7961268fb104bc1f4092129e7d9400841568a3204dc57a70', 'experiments/clip_patch_baseline/patch_beam.py': '53fcb5b22bb9f05d795fbd4f539cfd4ff0af39c1dea679f7df1a933edf2754be', 'experiments/clip_patch_baseline/score_full.py': 'fed170a6d12d1ab1bb969ae7eac29ee0855059417f6e2e876da193e279ca819a', 'experiments/clip_patch_baseline/patch_data.py': '3706ffec9c128c8fe12c4f21ba071939268fdd526ed614034cca8dfec55674d5', 'experiments/clip_patch_baseline/test_full_data.py': 'f47160cf9cd85c96a4495dfa8264c0cd5e7cd60e8d5b6a77022932399e557431', 'experiments/clip_patch_baseline/make_full_notebook.py': '2417f0c691e291808b9f3b44f8bdb60090fcaf8bb487d71948538a7d61d5679d', 'experiments/clip_patch_baseline/SMOKE_SOURCE.json': '6413058df1575ae8266c6f7ec29eea3cb60917147126bc55961de47c949abe49', 'experiments/clip_patch_baseline/FULL_XE10_PROTOCOL.md': 'a7c482bfe9efe3f23d4ebf235872204b8e0e0e9e69fae71e3a7b04b8980a986f', 'experiments/csa_paper/data.py': '5a5bb451dc20184d3ea597d4dc8415a120583571f4e1a4dac75e29a306213a71', 'experiments/csa_paper/real_bank_data.py': '2c791f8dd856198ec1954a2edd1f9e46520d422f20eff07220e462fc2c9bd9a0', 'experiments/csa_paper/cuda_runtime_guard.py': '640b5debc81607e3504930e4042571636cc3fbd525a6d0e5eb45abdd141fb773', 'experiments/csa_paper/saved_quality_data.py': 'f8a092506e3adf96292203190ea2f83de397d3028a3ddc6293d58fd360199a1e', 'experiments/clip_patch_baseline/local_stage.py': '76ec1d4ace24fe2d95ea396d489692b7ff3c6289b71fd88c81b110cd2ad9426b', 'experiments/clip_patch_baseline/test_local_stage.py': '028e8bff388c44123368dac2aa21887d589adf1e02bb2d3f7082870564b9cfbb', 'experiments/clip_patch_baseline/LOCAL_STAGE_V3_PROTOCOL.md': '83ba0fd376a2f1f6069eca1a082c954e8316b236fa343807f813a40eb0a9487d', 'experiments/clip_patch_baseline/semantic_data.py': 'a57af931f9e6be85f6efeee668b09fcca14c8a3bd54ebd695d9014ede2ab0540', 'experiments/clip_patch_baseline/semantic_patch_model.py': '10a385859ea1389a71f14a7a76e16a6989feb73fd581ca89659eab50ec53a0f3', 'experiments/clip_patch_baseline/semantic_training.py': '001f05f3e6959570571135a55abb7bc740217a447371bab627af8c72e0f26d11', 'experiments/clip_patch_baseline/test_semantic_data.py': 'e9d9132db2e83b47e7f29cd1c465389cb4f16db2cd62a1cf27f048a06e29a572', 'experiments/clip_patch_baseline/BANK_PROMPT_SCRATCH_PROTOCOL.md': '93d1ebef1d359d393323951a152f50d16559c0fd235dac27eda4940653399a0c', 'experiments/csa_paper/model.py': '139edbb1b73caf3dadaf5c960b25542e282f88baf2ce9bdf141de922ceacbc09', 'experiments/csa_paper/SAVE_BANK_V130_PACKAGE.json': '826ba36628d6587860e2d548e950ea953bd4bbb8c0a8dc64393089dbc1e058e1', 'experiments/csa_paper/lemma_correction_review_text/bank_words_corrected.json': 'cec76a0229f30361235a4b64e9712ac6bb2e79fc62a9241079a7af3fa95d13ac', 'experiments/csa_paper/FULL_PROMPT_FRESH_V175_RESULT.json': '452e94530b07b117be6873f004404b558e29e87cb09e97e27a1548d0bfd5f81a', 'experiments/clip_patch_baseline/memory_fusion.py': 'a70947e22263ddd27a7896a94cfb35cfff73102370288c53f86529964ea8f087', 'experiments/clip_patch_baseline/csa_full_data.py': '4028cd2b656d2244cabe2b4ab510b7a701bd8d462157adce62e10bf2060ca8c1', 'experiments/clip_patch_baseline/csa_full_runtime.py': '730a3282088480f56f8aaaee9af96a37a1bbb0c2f24256e19f14925f90c67486', 'experiments/clip_patch_baseline/run_csa_full.py': '86af1d1f9c2ed1352a7bc53b3f3d2fd8c00657b553dd34b870a2565e7efb072e', 'experiments/clip_patch_baseline/score_csa_full.py': '2a1ea2cf75dfe2ee400637d9094ccd504e06de0fb2dc0786e530bdee3e2528a5', 'experiments/clip_patch_baseline/test_csa_full_data.py': '6ab5adad7b3b5f5bbbb1648ef993e659c8309781ce5174bfb1971f6a4e72ec2c', 'experiments/clip_patch_baseline/csa_patch_model.py': '99b6441c4205b142256192cc3d840a905e00ecc1e730f30e1f5eb2c3cf8dc220', 'experiments/clip_patch_baseline/csa_sharing_data.py': '4d89428b8f833e34522bd51ec25640015edbebae4f0b5b81450058e2ebcb1243', 'experiments/clip_patch_baseline/gradient_agreement.py': 'dc06ffd922fccd9e14e860d07644dddcf320f7d9ac6410e3166cc4f53200971c', 'experiments/clip_patch_baseline/test_gradient_agreement.py': '2b2aa46091a5a752b3eab573f701c0f1e2849349d829e1c4d6a4d99ae73285a6', 'experiments/clip_patch_baseline/CSA_SHARED_FULL_XE10_PROTOCOL.md': '3b09530054a321f96691adb00cb4cb6340e42ce6eeba52b51268711c5df7ba0a', 'experiments/clip_patch_baseline/make_csa_full_notebook.py': '3cac3adbf862cbf7191aa36e0c89ae66c7337a4776eff73ce919d46f4455cfcc'}
VISUAL_CACHE=Path('/kaggle/input/datasets/ducanh2403/visual-cache')
COCO_JSON=Path('/kaggle/input/datasets/vuthetam/mscoco-2014/dataset_coco.json')
BANK_ROOT=Path('/kaggle/input/datasets/ducanh2403/csa-semantic-banks-v130')
PROMPT_ROOT=Path('/kaggle/input/datasets/ducanh2403/csa-prompt-full-karpathy-v175')
WALL_HOURS=3 # Parent stop cap, not ETA/billed quota. Do not auto-extend.
OUTPUT=Path('/kaggle/working/csa_hybrid_fp16_epoch1')
for p in (VISUAL_CACHE,COCO_JSON,BANK_ROOT,PROMPT_ROOT):
    if not p.exists():raise FileNotFoundError('Attach/correct inputs BEFORE GPU: '+str(p))
if OUTPUT.exists():raise RuntimeError('Preserve previous output; use a fresh runtime/output path')
if shutil.disk_usage('/tmp').free<4*1024**3 or shutil.disk_usage('/kaggle/working').free<8*1024**3:raise RuntimeError('Need 4 GiB temporary scratch and 8 GiB Working')
OUTPUT.mkdir(parents=True);SOURCE=Path(tempfile.mkdtemp(prefix='clip-patch-full10-',dir='/tmp'));DEPS=SOURCE/'deps';LOG=OUTPUT/'notebook.log'
PARENT_DEADLINE=time.monotonic()+WALL_HOURS*3600
os.environ['PYTHONPATH']=str(SOURCE);os.environ['OMP_NUM_THREADS']='1'
CONFIG=SOURCE/'config.json'
cfg=dict(csa_per_caption_full=True,semantic=True,bank_root=str(BANK_ROOT),prompt_root=str(PROMPT_ROOT),hybrid_sharing=True,profile_epoch=True,pilot_precision='fp16',prepare_seconds=1800,output=str(OUTPUT),stage=str(SOURCE/'stage'),dataset_json=str(COCO_JSON),visual_cache=str(VISUAL_CACHE),worker_seconds=600)
def write_config():
    CONFIG.write_text(json.dumps(cfg,indent=2));(OUTPUT/'protocol.json').write_text(json.dumps(cfg,indent=2))
write_config()
def download(url,expected,path):
    remaining=PARENT_DEADLINE-time.monotonic()
    if remaining<=0:raise TimeoutError('Parent cap during download')
    raw=urllib.request.urlopen(url,timeout=min(120,remaining)).read()
    if hashlib.sha256(raw).hexdigest()!=expected:raise ValueError('Changed pinned asset: '+path.name)
    path.parent.mkdir(parents=True,exist_ok=True);path.write_bytes(raw)

ACCEL_DEPS=SOURCE/'accelerate-deps'
os.environ['PYTHONPATH']=str(ACCEL_DEPS)+os.pathsep+str(SOURCE)

PILOT_SOURCES = {'run_csa_full.py': '"""USER-RUN CSA full10XE: Accelerate, image-grouped per-caption encoding, exact cursor/checkpoints."""\nimport argparse\nfrom datetime import timedelta\nimport importlib.metadata\nimport json\nimport os\nfrom pathlib import Path\nimport random\nimport shutil\nimport time\nimport numpy as np\nfrom real_bank_data import read,save,sha\nfrom full_data import prepare,load_stage,CaptionBatches,ImageBatches,loader,learning_rate,validate_state,STEPS,TRAIN_EVENTS,TRAIN_IMAGES,VALID_IMAGES\n\nfrom csa_full_data import HybridCaptionBatches,SharedCaptionBatches,order_for_epoch,current_epoch_projection\n\ndef atomic(path,value):\n    temp=path.with_suffix(\'.tmp\');save(temp,value);os.replace(temp,path)\n\ndef distributed(cfg,guard=False):\n    import torch\n    import torch.distributed as dist\n    from torch.nn import functional as F\n    from accelerate import Accelerator\n    from accelerate.utils import DistributedDataParallelKwargs,GradScalerKwargs,InitProcessGroupKwargs,DistributedType\n    import accelerate\n    from patch_beam import beam_search,text_from_ids\n    from full_runtime import tensor_digest\n    from csa_full_runtime import hybrid_logits,preflight,make_model\n    from semantic_data import SemanticContext\n    from cuda_runtime_guard import check_runtime\n    rank=int(os.environ[\'LOCAL_RANK\']);out=Path(cfg[\'output\']);check_runtime(out,rank)\n    if int(os.environ[\'WORLD_SIZE\'])!=2:raise ValueError(\'Select T4x2\')\n    torch.set_num_threads(1)\n    requested_precision=cfg.get(\'pilot_precision\',\'no\')\n    fp16_experiment=cfg.get(\'hybrid_sharing\') and cfg.get(\'pilot_batch_size\',32)==32 and (\n        cfg.get(\'pilot_updates\')==2000 or (cfg.get(\'pilot_updates\') is None and cfg.get(\'profile_epoch\')))\n    if requested_precision not in (\'no\',\'fp16\') or (requested_precision==\'fp16\' and\n       (not fp16_experiment or cfg.get(\'resume_root\'))):\n        raise ValueError(\'FP16 requires a fresh measured batch32 hybrid run\')\n    precision=\'no\' if guard else requested_precision\n    accelerator=Accelerator(mixed_precision=precision,gradient_accumulation_steps=1,dynamo_backend=\'no\',kwargs_handlers=[DistributedDataParallelKwargs(broadcast_buffers=False),GradScalerKwargs(init_scale=1024.),InitProcessGroupKwargs(backend=\'nccl\',timeout=timedelta(seconds=1200))])\n    device=accelerator.device\n    torch.backends.cuda.matmul.allow_tf32=False;torch.backends.cudnn.allow_tf32=False\n    started=time.monotonic();deadline=started+cfg[\'worker_seconds\']\n    hybrid=cfg.get(\'hybrid_sharing\',False)\n    profile=cfg.get(\'profile_epoch\',False)\n    epochs=1 if hybrid else 10\n    pilot_updates=cfg.get(\'pilot_updates\')\n    global_batch=cfg.get(\'pilot_batch_size\',32)\n    if global_batch not in (32,64) or (global_batch==64 and pilot_updates is None):\n        raise ValueError(\'Batch 64 is available only for the bounded throughput comparison\')\n    batch_scale=global_batch//32\n    if pilot_updates is not None and (not hybrid or type(pilot_updates) is not int or not 0<pilot_updates*batch_scale<STEPS or cfg.get(\'resume_root\')):\n        raise ValueError(\'A bounded throughput pilot requires a fresh hybrid run\')\n    try:\n        if accelerate.__version__!=\'1.15.0\' or accelerator.num_processes!=2 or accelerator.distributed_type!=DistributedType.MULTI_GPU or accelerator.gradient_accumulation_steps!=1 or accelerator.mixed_precision!=precision:raise ValueError(\'Unexpected Accelerate runtime\')\n        if precision==\'fp16\' and accelerator.scaler is None:raise ValueError(\'FP16 gradient scaler unavailable\')\n        save(out/f\'accelerate_runtime_rank{rank}.json\',dict(version=accelerate.__version__,device=str(device),precision=accelerator.mixed_precision,processes=accelerator.num_processes,accumulation=accelerator.gradient_accumulation_steps,prepared_dataloader=False))\n        data,words,stage=load_stage(cfg[\'stage\'],verify=rank==0);dist.barrier(device_ids=[rank])\n        if cfg.get(\'feature_storage\') and stage.get(\'feature_storage\')!=cfg[\'feature_storage\']:raise ValueError(\'Required local stage incomplete; do not fall back to mounted H5\')\n        torch.manual_seed(42)\n        if stage.get(\'semantic\',{}).get(\'method\')!=\'bank_prompt_patch_v1\':raise ValueError(\'Need complete full semantic stage\')\n        context=SemanticContext(cfg[\'stage\']);model=make_model(len(words),cfg).to(device)\n        wrapped=accelerator.prepare_model(model)\n        def predict(net,x,prefix,events):\n            return net(x,prefix,context.events(data,events,device))\n        def memory_for(x,indices):return model.encode(x,context.images(indices,device))\n        if guard:preflight(cfg,model,wrapped,data,words,rank,device,out,deadline,accelerator);return\n        guards=[read(out/f\'preflight_rank{r}.json\') for r in (0,1)]\n        if not all(g[\'passed\'] and g[\'initial_tensor_sha256\']==tensor_digest(model) for g in guards):raise ValueError(\'Fresh initialization/preflight mismatch\')\n        step_est=max(g[\'full_IO_step_p90_seconds\'] for g in guards);val_est=max(g[\'decode_seconds\']/g[\'decode_images\']*2500 for g in guards)\n        refs=read(Path(cfg[\'stage\'])/\'validation_references.json\')\n        identity=dict(schema=\'clip_patch_full_xe10_v1\',source=read(out/\'source_receipt.json\'),stage_files=stage[\'files_sha256\'],raw_sources=stage[\'sources\'],feature_profile=stage[\'feature_profile\'],\n            initial_sha256=guards[0][\'initial_tensor_sha256\'],global_batch=32,epochs=10,lr=\'warmup1epoch8e-6to1e-4_then_cosine_to1e-5\',dropout=.2,clip1=True,seed=42,framework=\'Accelerate\',sharing=False,sampler=\'image_grouped_42_plus_epoch\',\n            method=\'csa_structure_patch_prompt_grouped_per_caption_accelerate_v1\',software={k:importlib.metadata.version(k) for k in (\'torch\',\'numpy\',\'pycocoevalcap\',\'accelerate\')},test_used=False)\n        if hybrid:\n            identity.update(schema=\'csa_hybrid_epoch1_v1\',epochs=epochs,sharing=\'full_global32_only\',feature_storage=\'mounted_h5\')\n        if \'pilot_precision\' in cfg:identity.update(precision=precision)\n        if pilot_updates is not None:\n            identity.update(schema=\'csa_hybrid_throughput_v1\',pilot_updates=pilot_updates)\n            if global_batch==64:\n                identity.update(global_batch=global_batch,lr_index=\'caption_exposure\')\n        optimizer=accelerator.prepare_optimizer(torch.optim.Adam(model.parameters(),lr=8e-6,betas=(.9,.99),eps=1e-8,weight_decay=0))\n        state=dict(epoch=1,batch_cursor=0,phase=\'train\',updates=0,caption_events=0,history=[],best_epoch=None,best_CIDEr=None,epoch_nll_sum=0.,epoch_tokens=0)\n        random.seed(42+rank);np.random.seed(42+rank);torch.manual_seed(4242+rank);torch.cuda.manual_seed(4242+rank)\n        if cfg.get(\'resume_root\'):\n            root=Path(cfg[\'resume_root\']);receipt=read(root/\'checkpoint_receipt.json\')\n            if sha(root/\'checkpoint.bin\')!=receipt[\'sha256\']:raise ValueError(\'Changed resume checkpoint\')\n            ck=torch.load(root/\'checkpoint.bin\',map_location=\'cpu\',weights_only=False)\n            if ck[\'identity\']!=identity:raise ValueError(\'Resume recipe/data/source/software differs\')\n            state=ck[\'state\'];validate_state(state);model.load_state_dict(ck[\'model\'],strict=True);optimizer.load_state_dict(ck[\'optimizer\'])\n            rng=ck[\'rng_by_rank\'][rank];random.setstate(rng[\'python\']);np.random.set_state(rng[\'numpy\']);torch.set_rng_state(rng[\'torch\']);torch.cuda.set_rng_state(rng[\'cuda\'],device)\n            if rank==0:\n                for row in state[\'history\']:\n                    folder=root/\'evaluations\'/f"epoch_{row[\'epoch\']:02d}"\n                    for name,h in row[\'files_sha256\'].items():\n                        if sha(folder/name)!=h:raise ValueError(\'Changed resume validation \'+name)\n                    shutil.copytree(folder,out/\'evaluations\'/folder.name)\n                if state[\'best_epoch\'] is not None:\n                    b=read(root/\'best_receipt.json\') if (root/\'best_receipt.json\').is_file() else {}\n                    if b.get(\'epoch\')!=state[\'best_epoch\'] and state[\'batch_cursor\']==0 and state[\'phase\']==\'train\' and state[\'best_epoch\']==state[\'epoch\']-1:\n                        shutil.copy2(root/\'checkpoint.bin\',out/\'best.bin\');save(out/\'best_receipt.json\',dict(sha256=receipt[\'sha256\'],epoch=state[\'best_epoch\'],CIDEr=state[\'best_CIDEr\'],test_used=False))\n                    else:\n                        if b.get(\'epoch\')!=state[\'best_epoch\'] or sha(root/\'best.bin\')!=b[\'sha256\']:raise ValueError(\'Changed resume best\')\n                        for name in (\'best.bin\',\'best_receipt.json\'):shutil.copy2(root/name,out/name)\n                if receipt.get(\'training_log_sha256\'):\n                    if sha(root/\'training.jsonl\')!=receipt[\'training_log_sha256\']:raise ValueError(\'Changed resume training log\')\n                    shutil.copy2(root/\'training.jsonl\',out/\'training.jsonl\')\n            del ck;dist.barrier(device_ids=[rank])\n        if rank==0:save(out/\'identity.json\',identity)\n        def checkpoint(promote=False):\n            if global_batch==32:\n                validate_state(state)\n            elif (state[\'epoch\']!=1 or state[\'phase\']!=\'train\' or state[\'updates\']!=state[\'batch_cursor\']\n                  or state[\'caption_events\']!=global_batch*state[\'updates\'] or state[\'history\']):\n                raise ValueError(\'Batch 64 pilot checkpoint exposure mismatch\')\n            rng=dict(python=random.getstate(),numpy=np.random.get_state(),torch=torch.get_rng_state(),cuda=torch.cuda.get_rng_state(device));all_rng=[None,None];dist.all_gather_object(all_rng,rng)\n            if rank==0:\n                payload=dict(model=model.state_dict(),optimizer=optimizer.state_dict(),state=state,identity=identity,rng_by_rank=all_rng)\n                if accelerator.scaler is not None:payload[\'scaler\']=accelerator.scaler.state_dict()\n                temp=out/\'checkpoint.tmp\';torch.save(payload,temp);os.replace(temp,out/\'checkpoint.bin\')\n                receipt=dict(sha256=sha(out/\'checkpoint.bin\'),updates=state[\'updates\'],epoch=state[\'epoch\'],phase=state[\'phase\'],training_log_sha256=sha(out/\'training.jsonl\') if (out/\'training.jsonl\').is_file() else None)\n                atomic(out/\'checkpoint_receipt.json\',receipt);atomic(out/\'progress.json\',state)\n                if promote:\n                    shutil.copy2(out/\'checkpoint.bin\',out/\'best.tmp\');os.replace(out/\'best.tmp\',out/\'best.bin\')\n                    atomic(out/\'best_receipt.json\',dict(sha256=receipt[\'sha256\'],epoch=state[\'best_epoch\'],CIDEr=state[\'best_CIDEr\'],test_used=False))\n            dist.barrier(device_ids=[rank])\n        if pilot_updates is None:\n            remaining_steps=epochs*STEPS-state[\'updates\'];remaining_epochs=epochs+1-state[\'epoch\']\n            projected=current_epoch_projection(state,step_est,val_est)\n            if rank==0:save(out/\'measured_projection.json\',dict(remaining_steps=remaining_steps,remaining_epochs=remaining_epochs,seconds=projected,available=deadline-time.monotonic(),not_guaranteed=True,scope=\'remaining current epoch + VALID + reserve\'))\n            if not profile and projected>deadline-time.monotonic():checkpoint();raise TimeoutError(\'Measured current epoch+VALID projection exceeds cap; initial checkpoint saved, resume explicitly\')\n        def validate(epoch):\n            validation_start=time.monotonic()\n            wrapped.eval();folder=out/\'evaluations\'/f\'epoch_{epoch:02d}\';folder.mkdir(parents=True,exist_ok=True);rows=[];nll_sum=0.;tokens=0\n            images=np.arange(TRAIN_IMAGES,TRAIN_IMAGES+VALID_IMAGES)[rank::2];gate_stats=None\n            with torch.inference_mode():\n                for x,index in loader(ImageBatches(cfg[\'stage\'],images)):\n                    if time.monotonic()>deadline-60:raise TimeoutError(\'VALID decode deadline; preserve training checkpoint\')\n                    xx=x.to(device,non_blocking=True)\n                    if cfg.get(\'semantic_strength_gate\'):\n                        from memory_fusion import update_gate_stats\n                        memory,alpha=model.encode_with_gate(xx,context.images(index,device));gate_stats=update_gate_stats(gate_stats,alpha.cpu().numpy())\n                    else:memory=memory_for(xx,index)\n                    seq,scores=beam_search(model,memory)\n                    for i,s,v in zip(index.tolist(),seq[:,0].cpu().tolist(),scores[:,0].cpu().tolist()):rows.append(dict(coco_id=int(data[\'ids\'][i]),caption=text_from_ids(s,words),ids=s,score=float(v)))\n                ev=np.flatnonzero(np.isin(data[\'caption_image\'],images))\n                decode_seconds=time.monotonic()-validation_start\n                for x,seq,event_ids in loader(CaptionBatches(cfg[\'stage\'],ev,rank=None)):\n                    if time.monotonic()>deadline-60:raise TimeoutError(\'VALID NLL deadline\')\n                    x=x.to(device,non_blocking=True);seq=seq.to(device,non_blocking=True)\n                    if precision==\'fp16\':\n                        # Accelerate wraps forward with autocast. Direct encode/decode preserves FP32 VALID.\n                        memory=model.encode(x,context.events(data,event_ids,device))\n                        logits=model.decode(memory,seq[:,:-1])\n                        if logits.dtype!=torch.float32:raise ValueError(\'Expected FP32 VALID logits\')\n                    else:\n                        logits=predict(model,x,seq[:,:-1],event_ids)\n                    nll_sum+=float(F.cross_entropy(logits.flatten(0,1),seq[:,1:].flatten(),ignore_index=0,reduction=\'sum\'));tokens+=int((seq[:,1:]!=0).sum())\n            if len(rows)!=2500:raise ValueError(\'Incomplete VALID worker\')\n            save(folder/f\'val_rank{rank}.json\',rows)\n            worker=dict(complete=True,epoch=epoch,images=2500,NLL_sum=nll_sum,NLL_tokens=tokens,decode_seconds=decode_seconds,NLL_seconds=time.monotonic()-validation_start-decode_seconds)\n            if cfg.get(\'semantic_strength_gate\'):\n                if gate_stats[\'count\']!=2500*196:raise ValueError(\'Incomplete VALID strength diagnostics\')\n                worker[\'semantic_strength_accumulator\']=gate_stats\n            save(folder/f\'worker{rank}.json\',worker)\n            dist.barrier(device_ids=[rank]);point=[None]\n            if rank==0:\n                scoring_start=time.monotonic()\n                from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer\n                from pycocoevalcap.cider.cider import Cider\n                all_rows=read(folder/\'val_rank0.json\')+read(folder/\'val_rank1.json\')\n                if len({r[\'coco_id\'] for r in all_rows})!=5000 or {r[\'coco_id\'] for r in all_rows}!={r[\'coco_id\'] for r in refs}:raise ValueError(\'VALID IDs differ\')\n                ptb=PTBTokenizer();gts=ptb.tokenize({r[\'coco_id\']:[dict(caption=t) for t in r[\'references\']] for r in refs});hyp=ptb.tokenize({r[\'coco_id\']:[dict(caption=r[\'caption\'])] for r in all_rows})\n                cider,per=Cider().compute_score(gts,hyp)\n                if not np.isfinite(cider):raise ValueError(\'Nonfinite CIDEr\')\n                save(folder/\'cider_per_image.json\',[dict(coco_id=i,CIDEr=float(s)) for i,s in zip(gts,per)])\n                workers=[read(folder/f\'worker{r}.json\') for r in (0,1)]\n                value=dict(epoch=epoch,CIDEr=float(cider),mean_words=float(np.mean([len(r[\'caption\'].split()) for r in all_rows])),short_le2=sum(len(r[\'caption\'].split())<=2 for r in all_rows),NLL=sum(w[\'NLL_sum\'] for w in workers)/sum(w[\'NLL_tokens\'] for w in workers),TRAIN_online_NLL=state[\'epoch_nll_sum\']/state[\'epoch_tokens\'])\n                if profile:\n                    value.update(CIDEr_scoring_seconds=time.monotonic()-scoring_start,validation_wall_seconds=time.monotonic()-validation_start)\n                if cfg.get(\'semantic_strength_gate\'):\n                    from memory_fusion import merge_gate_stats\n                    value[\'semantic_strength\']=merge_gate_stats([w[\'semantic_strength_accumulator\'] for w in workers])\n                save(folder/\'result.json\',value);value[\'files_sha256\']={p.name:sha(p) for p in folder.iterdir() if p.is_file()};point[0]=value\n                print(\'FULL VALID\',value,flush=True)\n            dist.broadcast_object_list(point,src=0);return point[0]\n        while state[\'epoch\']<=epochs:\n            epoch=state[\'epoch\'];wrapped.train();order=order_for_epoch(data,epoch);cursor=state[\'batch_cursor\'];speed_start=None\n            if state[\'phase\']==\'train\':\n                train_start=time.monotonic()\n                encoded_rows=0\n                skipped_steps=0\n                nonfinite_norms=0\n                batch_type=HybridCaptionBatches if hybrid else CaptionBatches\n                if global_batch==64:\n                    batch_type=SharedCaptionBatches\n                    order=order[:pilot_updates*global_batch]\n                it=iter(loader(batch_type(cfg[\'stage\'],order,rank,cursor,batch_size=global_batch)))\n                wait_seconds=0.\n                samples=[]\n                window_start=time.monotonic()\n                stop_reserve=120 if pilot_updates is not None else 900\n                for offset in range(cursor,pilot_updates if pilot_updates is not None else STEPS):\n                    stop=torch.tensor(int(time.monotonic()>deadline-stop_reserve),device=device);dist.all_reduce(stop,op=dist.ReduceOp.MAX)\n                    if int(stop):checkpoint();raise TimeoutError(\'Saved before parent/worker cap\')\n                    fetch_start=time.monotonic()\n                    x,seq,ev=next(it)\n                    wait_seconds+=time.monotonic()-fetch_start\n                    events=None\n                    if profile and (offset+1)%20==0:\n                        events=[torch.cuda.Event(enable_timing=True) for _ in range(3)]\n                        events[0].record()\n                    x=x.to(device,non_blocking=True);seq=seq.to(device,non_blocking=True)\n                    if events is not None:events[1].record()\n                    lr=learning_rate(state[\'updates\']*batch_scale+1)\n                    for g in optimizer.param_groups:g[\'lr\']=lr\n                    share=hybrid and (offset+1)*global_batch<=len(order)\n                    if pilot_updates is not None and len(ev)!=global_batch//2:\n                        raise ValueError(\'Incomplete rank-local training batch\')\n                    if hybrid and not share:\n                        expected_tail=16 if rank==0 else 11\n                        if len(x)!=expected_tail or len(ev)!=expected_tail:raise ValueError(\'Expected ordinary global tail27 split 16+11\')\n                    encoded_rows+=len(x)\n                    optimizer.zero_grad(set_to_none=True);logits=hybrid_logits(wrapped,x,seq[:,:-1],ev,data,context,device,share);count=torch.tensor([int((seq[:,1:]!=0).sum()),len(ev)],device=device);dist.all_reduce(count)\n                    nll=F.cross_entropy(logits.flatten(0,1),seq[:,1:].flatten(),ignore_index=0,reduction=\'sum\');loss=nll*(2/int(count[0]))\n                    if not torch.isfinite(loss):raise ValueError(\'Nonfinite XE\')\n                    accelerator.backward(loss);norm=accelerator.clip_grad_norm_(model.parameters(),1)\n                    finite_norm=bool(torch.isfinite(norm))\n                    if not finite_norm and precision!=\'fp16\':raise ValueError(\'Nonfinite gradient norm\')\n                    optimizer.step()\n                    step_skipped=accelerator.optimizer_step_was_skipped\n                    if not finite_norm and not step_skipped:raise ValueError(\'Nonfinite gradient norm without a scaler skip\')\n                    skipped_steps+=int(step_skipped)\n                    nonfinite_norms+=int(not finite_norm)\n                    if events is not None:\n                        events[2].record()\n                        samples.append(events)\n                    total=nll.detach().double();dist.all_reduce(total)\n                    state.update(batch_cursor=offset+1,updates=state[\'updates\']+1,caption_events=(epoch-1)*TRAIN_EVENTS+min((offset+1)*global_batch,TRAIN_EVENTS),epoch_nll_sum=state[\'epoch_nll_sum\']+float(total),epoch_tokens=state[\'epoch_tokens\']+int(count[0]))\n                    if ((offset+1)*batch_scale)%100==0 or offset+1==STEPS:\n                        if profile:\n                            if samples:samples[-1][2].synchronize()\n                            timing=dict(end_update=offset+1,wall_seconds=time.monotonic()-window_start,\n                                        batch_wait_seconds=wait_seconds,sampled_steps=len(samples),\n                                        patch_token_copy_ms=sum(a.elapsed_time(b) for a,b,c in samples),\n                                        training_step_ms=sum(b.elapsed_time(c) for a,b,c in samples))\n                            with (out/f\'train_profile_rank{rank}.jsonl\').open(\'a\',encoding=\'utf-8\') as f:\n                                f.write(json.dumps(timing)+\'\\n\')\n                            wait_seconds=0.\n                            samples.clear()\n                            window_start=time.monotonic()\n                        if rank==0:\n                            report=dict(epoch=epoch,updates=state[\'updates\'],batch_cursor=offset+1,caption_events=state[\'caption_events\'],NLL=float(total)/int(count[0]),lr=lr,grad_norm=float(norm) if finite_norm else None,seconds=time.monotonic()-started,train_seconds=time.monotonic()-train_start,\n                                        skipped_steps=skipped_steps,nonfinite_norms=nonfinite_norms)\n                            with (out/\'training.jsonl\').open(\'a\',encoding=\'utf-8\') as f:f.write(json.dumps(report)+\'\\n\')\n                            print(\'FULL XE\',report,flush=True)\n                        completed=offset+1-cursor\n                        if completed==100:speed_start=time.monotonic()\n                        if not profile and pilot_updates is None and completed>=200 and speed_start is not None:\n                            measured=(time.monotonic()-speed_start)/(completed-100);remaining=current_epoch_projection(state,measured,val_est)\n                            stop=torch.tensor(int(remaining>deadline-time.monotonic()),device=device);dist.all_reduce(stop,op=dist.ReduceOp.MAX)\n                            if int(stop):checkpoint();raise TimeoutError(\'Current epoch+VALID projection exceeds remaining cap; saved for explicitresume\')\n                    if (state[\'updates\']*batch_scale)%500==0 or state[\'updates\']==pilot_updates:checkpoint()\n                if pilot_updates is not None:\n                    if state[\'updates\']!=pilot_updates or state[\'caption_events\']!=global_batch*pilot_updates:\n                        raise ValueError(\'Incorrect bounded training exposure\')\n                    del it\n                    timing=dict(seconds=time.monotonic()-train_start,updates=pilot_updates,encoded_rows=encoded_rows,\n                                peak_allocated_GiB=torch.cuda.max_memory_allocated(device)/1024**3,\n                                precision=precision,skipped_steps=skipped_steps,nonfinite_norms=nonfinite_norms,\n                                optimizer_steps=pilot_updates-skipped_steps)\n                    save(out/f\'train_timing_rank{rank}.json\',timing)\n                    dist.barrier(device_ids=[rank])\n                    if rank==0:\n                        save(out/\'training_complete.json\',dict(status=\'CSA_HYBRID_2000_COMPLETE\',**state,test_used=False))\n                    return\n                del it;state[\'phase\']=\'validate\';checkpoint()\n                if hybrid:\n                    save(out/f\'train_timing_rank{rank}.json\',dict(seconds=time.monotonic()-train_start,updates=STEPS-cursor,\n                         shared_updates=STEPS-cursor-1,ordinary_tail_updates=1,encoded_rows=encoded_rows,\n                         peak_allocated_GiB=torch.cuda.max_memory_allocated(device)/1024**3,\n                         precision=precision,skipped_steps=skipped_steps,nonfinite_norms=nonfinite_norms,\n                         optimizer_steps=STEPS-cursor-skipped_steps))\n            point=validate(epoch);better=state[\'best_CIDEr\'] is None or point[\'CIDEr\']>state[\'best_CIDEr\']\n            if better:state.update(best_CIDEr=point[\'CIDEr\'],best_epoch=epoch)\n            state[\'history\'].append(point);state.update(epoch=epoch+1,batch_cursor=0,phase=\'train\',epoch_nll_sum=0.,epoch_tokens=0);checkpoint(promote=better)\n        if rank==0:save(out/\'training_complete.json\',dict(status=\'CSA_HYBRID_EPOCH1_COMPLETE\' if hybrid else \'PATCH_FULL_XE10_TRAINING_COMPLETE\',**state,worker_seconds=time.monotonic()-started,timing_scope=\'this worker invocation only; excludes notebook setup and earlier resume segments\'))\n    finally:\n        accelerator.end_training()\n        if dist.is_initialized():dist.destroy_process_group()\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();p.add_argument(\'mode\',choices=(\'prepare\',\'preflight\',\'worker\'));p.add_argument(\'--config\',required=True)\n    a=p.parse_args();cfg=read(a.config)\n    if a.mode==\'prepare\':\n        deadline=time.monotonic()+cfg.get(\'prepare_seconds\',1800)\n        if cfg.get(\'semantic\'):\n            from semantic_data import checked_inputs\n            checked_inputs(cfg)\n            print(\'Frozen bank130/prompt175 exact source precheck PASS before patch copy.\',flush=True)\n        prepare(cfg)\n        if cfg.get(\'feature_storage\')==\'local_fp16_memmap_v2\':\n            from local_stage import stage_local\n            stage_local(cfg,deadline)\n        if cfg.get(\'semantic\'):\n            from semantic_data import stage_semantics\n            stage_semantics(cfg,deadline)\n    else:distributed(cfg,guard=a.mode==\'preflight\')\n', 'csa_full_runtime.py': '"""Full-data Accelerate guards/benchmark; neural calls only in user\'s worker."""\nimport copy\nfrom pathlib import Path\nimport time\nimport numpy as np\nfrom real_bank_data import save\nfrom full_runtime import tensor_digest\nfrom csa_full_data import order_for_epoch\nfrom full_data import CaptionBatches,ImageBatches,loader\nfrom csa_sharing_data import local_events\nfrom gradient_agreement import gradient_agreement\n\n\ndef make_model(vocab,cfg):\n    import torch\n    from csa_patch_model import CSAPatchCaptioner\n    if not cfg.get(\'csa_per_caption_full\') or cfg.get(\'semantic_strength_gate\'):\n        raise ValueError(\'Need CSA per-caption configuration, no old strength gate\')\n    banks=[torch.from_numpy(np.load(Path(cfg[\'stage\'])/(\'sem_\'+k+\'_bank.npy\'),allow_pickle=False)) for k in (\'noun\',\'verb\')]\n    if cfg.get(\'hybrid_sharing\'):\n        from csa_shared_model import SharedCSAPatchCaptioner\n        return SharedCSAPatchCaptioner(vocab,*banks)\n    return CSAPatchCaptioner(vocab,*banks)\n\n\ndef hybrid_logits(net, x, prefix, events, data, context, device, share):\n    import torch\n    from csa_sharing_data import image_plan\n    if not share:\n        return net(x, prefix, context.events(data, events, device))\n    event_ids = np.asarray(events)\n    first, inverse = image_plan(data[\'caption_image\'], event_ids, True)\n    if len(x) != len(first):\n        raise ValueError(\'Shared feature rows differ from local image plan\')\n    images = data[\'caption_image\'][event_ids[first]]\n    return net(x, prefix, context.images(images, device), torch.as_tensor(inverse, device=device))\n\n\ndef preflight(cfg,model,wrapped,data,words,rank,device,out,deadline,accelerator):\n    import torch\n    import torch.distributed as dist\n    from torch.nn import functional as F\n    from semantic_data import SemanticContext\n    from patch_beam import beam_search,check_paths\n    context=SemanticContext(cfg[\'stage\']);order=order_for_epoch(data,1);model.eval();initial=tensor_digest(model)\n    def inputs(events):\n        ds=CaptionBatches(cfg[\'stage\'],events)\n        x,seq,ev=ds[0];ds.reader.close()\n        return torch.from_numpy(x).to(device),torch.from_numpy(seq).to(device),context.events(data,ev,device)\n    parity=[]\n    for label,events in [(\'first32\',order[:32]),(\'tail27\',order[-27:])]:\n        denominator=int((data[\'sequences\'][events,1:]!=0).sum());reference_grads=None;reference_nll=None\n        if rank==0:\n            reference=copy.deepcopy(model).eval();reference_nll=0.\n            for rr in (0,1):\n                x,seq,p=inputs(local_events(events,rr));logits=reference(x,seq[:,:-1],p)\n                loss=F.cross_entropy(logits.flatten(0,1),seq[:,1:].flatten(),ignore_index=0,reduction=\'sum\')/denominator\n                loss.backward();reference_nll+=float(loss.detach())\n                del x,seq,p,logits,loss\n            reference_grads={n:v.grad.detach().cpu().clone() for n,v in reference.named_parameters()};del reference\n        ev=local_events(events,rank);x,seq,p=inputs(ev)\n        with torch.no_grad():\n            expected=wrapped(x,seq[:,:-1],p);changed=seq[:,:-1].clone();changed[:,3]=(changed[:,3]+7)%len(words)\n            torch.testing.assert_close(expected[:,:3],model(x,changed,p)[:,:3],rtol=1e-4,atol=5e-5)\n        wrapped.zero_grad(set_to_none=True);logits=wrapped(x,seq[:,:-1],p)\n        nll=F.cross_entropy(logits.flatten(0,1),seq[:,1:].flatten(),ignore_index=0,reduction=\'sum\')\n        total=torch.tensor(int((seq[:,1:]!=0).sum()),device=device);dist.all_reduce(total)\n        if int(total)!=denominator:raise ValueError(\'Wrong full-data globaltokens\')\n        accelerator.backward(nll*(2/denominator))\n        stats=torch.tensor([float(nll.detach()),len(x),len(seq)],dtype=torch.float64,device=device);dist.all_reduce(stats)\n        error=[None];diagnostics={}\n        if rank==0:\n            try:\n                if abs(float(stats[0])/denominator-reference_nll)>5e-5:raise ValueError(\'Full-data reference NLL differs\')\n                for name,v in model.named_parameters():\n                    if v.grad is None or not torch.isfinite(v.grad).all():raise ValueError(\'Missing/nonfinitegradient \'+name)\n                    diagnostics[name]=gradient_agreement(v.grad.detach().cpu().numpy(),reference_grads[name].numpy())\n                failed=[n for n,d in diagnostics.items() if not d[\'passed\']]\n                if failed:raise ValueError(\'Full-data gradient parity \'+\',\'.join(failed))\n            except BaseException as exc:error[0]=str(exc)\n            parity.append(dict(case=label,passed=error[0] is None,error=error[0],NLL=float(stats[0])/denominator,reference_NLL=reference_nll,\n                               global_tokens=denominator,caption_events=int(stats[2]),encoded_images=int(stats[1]),gradient_diagnostics=diagnostics))\n            save(out/\'full_parity.json\',parity)\n        dist.broadcast_object_list(error,src=0)\n        if error[0]:raise ValueError(error[0])\n        del x,seq,p,logits,nll,reference_grads,expected,changed\n    guard_peak=torch.cuda.max_memory_allocated(device)/1024**3;wrapped.zero_grad(set_to_none=True)\n    optimizer=accelerator.prepare_optimizer(torch.optim.Adam(model.parameters(),lr=1e-4,betas=(.9,.99),eps=1e-8,weight_decay=0))\n    wrapped.train();times=[];encoded=[];torch.cuda.reset_peak_memory_stats(device)\n    batch_type=CaptionBatches\n    if cfg.get(\'hybrid_sharing\'):\n        from csa_full_data import HybridCaptionBatches\n        batch_type=HybridCaptionBatches\n    ds=batch_type(cfg[\'stage\'],order[:64*32],rank)\n    iterator=iter(loader(ds))\n    for _ in range(64):\n        if time.monotonic()>deadline-120:raise TimeoutError(\'Full data preflight cap\')\n        torch.cuda.synchronize(device);started=time.monotonic()\n        x,seq,events=next(iterator)\n        x=x.to(device,non_blocking=True);seq=seq.to(device,non_blocking=True)\n        optimizer.zero_grad(set_to_none=True)\n        logits=hybrid_logits(wrapped,x,seq[:,:-1],events,data,context,device,cfg.get(\'hybrid_sharing\'));count=torch.tensor(int((seq[:,1:]!=0).sum()),device=device);dist.all_reduce(count)\n        loss=F.cross_entropy(logits.flatten(0,1),seq[:,1:].flatten(),ignore_index=0,reduction=\'sum\')*(2/int(count))\n        accelerator.backward(loss);norm=accelerator.clip_grad_norm_(model.parameters(),1)\n        if not torch.isfinite(loss) or not torch.isfinite(norm):raise ValueError(\'Nonfinite preflighttrain\')\n        optimizer.step();torch.cuda.synchronize(device);times.append(time.monotonic()-started);encoded.append(len(x))\n    if len(times)!=64:raise ValueError(\'Incomplete64step preflight\')\n    del optimizer,ds,iterator,x,seq,events,logits,loss\n    probe=copy.deepcopy(model).eval();indices=np.arange(64)[rank::2]\n    with torch.inference_mode():\n        # Check real beam scoring before altering ONLY the separate timing copy.\n        images=ImageBatches(cfg[\'stage\'],indices[:2]);xx,ii=images[0];images.reader.close()\n        memory=probe.encode(torch.from_numpy(xx).to(device),context.images(ii,device));seq,scores=beam_search(probe,memory);check_paths(probe,memory,seq,scores)\n        probe.head.bias[2]=-1e9\n        torch.cuda.synchronize(device);seen=0;decode_times=[]\n        batches=ImageBatches(cfg[\'stage\'],indices);iterator=iter(loader(batches))\n        for batch_index in range(len(batches)):\n            if time.monotonic()>deadline-30:raise TimeoutError(\'Forced beam preflight cap\')\n            began=time.monotonic();xx,ii=next(iterator)\n            memory=probe.encode(xx.to(device),context.images(ii,device));seq,scores=beam_search(probe,memory)\n            if seq.shape[-1]!=21 or (seq[:,:,1:]==2).any():raise ValueError(\'Forced20token timing not realized\')\n            torch.cuda.synchronize(device)\n            if batch_index:seen+=len(ii);decode_times.append(time.monotonic()-began)\n        decode_seconds=sum(decode_times);del batches,iterator\n    save(out/f\'preflight_rank{rank}.json\',dict(passed=True,rank=rank,initial_tensor_sha256=initial,discarded_benchmark_updates=64,\n        causal=True,ordinary_DDP_gradient_parity=True,beam_score_rescore=True,forced20token_decode=True,\n        full_IO_step_median_seconds=float(np.median(times[16:])),full_IO_step_p90_seconds=float(np.quantile(times[16:],.9)),\n        mean_encoded_images_local=float(np.mean(encoded)),decode_images=seen,decode_seconds=decode_seconds,\n        peak_allocated_GiB=torch.cuda.max_memory_allocated(device)/1024**3,gradient_guard_peak_GiB=guard_peak,\n        limits=\'TRAIN-only sampled full-loader preflight; forcedEOS-suppressed beam timing copy, not captions/VALID quality/billing.64updates discarded, mainworker recreated fresh.\'))\n    del probe;dist.barrier(device_ids=[rank])\n', 'csa_full_data.py': '"""Full TRAIN image-grouped order and unique-image loader; NumPy only."""\nimport numpy as np\nimport full_data as base\nfrom csa_sharing_data import image_plan\n\n\ndef grouped_order(caption_image, train_events, train_images, epoch):\n    images=np.asarray(caption_image[:train_events])\n    if type(epoch)!=int or not 1<=epoch<=10 or len(images)!=train_events or (np.diff(images)<0).any() or not np.array_equal(np.unique(images),np.arange(train_images)):\n        raise ValueError(\'Invalid epoch or incomplete/sorted TRAIN caption mapping\')\n    boundaries=np.r_[0,np.flatnonzero(np.diff(images))+1,len(images)]\n    return np.concatenate([np.arange(boundaries[i],boundaries[i+1],dtype=np.int64) for i in np.random.default_rng(42+epoch).permutation(train_images)])\n\n\ndef order_for_epoch(data,epoch):\n    return grouped_order(data[\'caption_image\'],base.TRAIN_EVENTS,base.TRAIN_IMAGES,epoch)\n\n\nclass SharedCaptionBatches(base.CaptionBatches):\n    def __getitem__(self,item):\n        if self.reader is None:self.reader=base.FeatureReader(self.root)\n        start=(item+self.cursor)*self.batch_size;events=self.events[start:start+self.batch_size]\n        if self.rank is not None:\n            local_size=self.batch_size//2\n            events=events[self.rank*local_size:(self.rank+1)*local_size]\n        if not len(events):raise ValueError(\'Empty distributed batch\')\n        data=self.reader.data;first,_=image_plan(data[\'caption_image\'],events,True)\n        images=data[\'caption_image\'][events[first]]\n        features=np.stack([self.reader.get(i) for i in images]).astype(np.float32)\n        seq=np.asarray(data[\'sequences\'][events],np.int64);seq=seq[:,:int((seq!=0).sum(1).max())]\n        return features,seq,events.copy()\n\n\nclass HybridCaptionBatches(SharedCaptionBatches):\n    def __getitem__(self, item):\n        if self.rank not in (0, 1) or self.batch_size != 32:\n            raise ValueError(\'Hybrid sharing requires two training ranks and global batch 32\')\n        if (item + self.cursor + 1) * self.batch_size > len(self.events):\n            return base.CaptionBatches.__getitem__(self, item)\n        return super().__getitem__(item)\n\n\ndef current_epoch_projection(state,step_seconds,valid_seconds):\n    if state.get(\'epoch\',1)==11:return 900\n    steps=0 if state[\'phase\']==\'validate\' else base.STEPS-state[\'batch_cursor\']\n    return 1.35*(steps*step_seconds+valid_seconds+90)+900\n', 'csa_shared_model.py': '"""Same CSA parameters/decoder; repeat encoded memories only after image encoding."""\nfrom csa_patch_model import CSAPatchCaptioner\n\n\nclass SharedCSAPatchCaptioner(CSAPatchCaptioner):\n    def forward(self, patches, prefix, p, inverse=None):\n        memory = self.encode(patches, p)\n        if inverse is not None:\n            memory = memory.index_select(0, inverse)\n        return self.decode(memory, prefix)\n', 'test_csa_full_data.py': '"""Author-safe full grouped sampler/collation/cursor/integration checks."""\nimport ast\nfrom pathlib import Path\nfrom unittest import TestCase,main\nfrom unittest.mock import patch\nimport numpy as np\nimport csa_full_data as d\nimport full_data as base\n\nHERE=Path(__file__).resolve().parent\n\n\nclass Reader:\n    def __init__(self,root):\n        images=np.repeat(np.arange(7),[5,4,6,3,5,2,2])\n        seq=np.array([[1,4+i%4,2,0] for i in range(27)],np.int32)\n        self.data=dict(caption_image=images,sequences=seq);self.read=[]\n    def get(self,i):self.read.append(int(i));return np.full((196,768),i,np.float16)\n\n\nclass Checks(TestCase):\n    def test_all_full_events_once_deterministic_and_tail(self):\n        # Synthetic unequal group sizes with the same fullTRAIN event total.\n        mapping=np.repeat(np.arange(base.TRAIN_IMAGES),np.r_[np.full(312,6),np.full(base.TRAIN_IMAGES-312,5)])\n        self.assertEqual(len(mapping),base.TRAIN_EVENTS)\n        order=d.grouped_order(mapping,len(mapping),base.TRAIN_IMAGES,1)\n        np.testing.assert_array_equal(np.sort(order),np.arange(base.TRAIN_EVENTS))\n        np.testing.assert_array_equal(order,d.grouped_order(mapping,len(mapping),base.TRAIN_IMAGES,1))\n        self.assertFalse(np.array_equal(order,d.grouped_order(mapping,len(mapping),base.TRAIN_IMAGES,2)))\n        self.assertEqual(len(order[-(len(order)%32):]),27)\n        with self.assertRaises(ValueError):d.grouped_order(np.r_[mapping[:-1],base.TRAIN_IMAGES],len(mapping),base.TRAIN_IMAGES,1)\n\n    def test_unique_collation_caption_order_tail_and_cursor(self):\n        for rank,expected in ((0,16),(1,11)):\n            with patch.object(base,\'FeatureReader\',Reader):\n                ds=d.SharedCaptionBatches(\'unused\',np.arange(27),rank=rank)\n                features,seq,events=ds[0]\n                self.assertEqual(len(seq),expected);self.assertEqual(len(seq),len(events))\n                self.assertEqual(len(features),len(set(ds.reader.data[\'caption_image\'][events])))\n                self.assertEqual(len(ds.reader.read),len(features))\n                np.testing.assert_array_equal(seq[:,1],ds.reader.data[\'sequences\'][events,1])\n        with patch.object(base,\'FeatureReader\',Reader):\n            ds=d.SharedCaptionBatches(\'unused\',np.arange(27),cursor=1,batch_size=16)\n            self.assertEqual(ds[0][2].tolist(),list(range(16,27)))\n\n    def test_horizon_and_progressive_epoch_budget(self):\n        self.assertEqual(base.STEPS,17711)\n        self.assertAlmostEqual(base.learning_rate(1),8e-6)\n        self.assertAlmostEqual(base.learning_rate(10*base.STEPS),1e-5)\n        state=dict(phase=\'train\',batch_cursor=base.STEPS-100)\n        estimate=d.current_epoch_projection(state,.2,200)\n        self.assertAlmostEqual(estimate,1.35*(100*.2+200+90)+900)\n        state.update(phase=\'validate\',batch_cursor=base.STEPS)\n        self.assertAlmostEqual(d.current_epoch_projection(state,.2,200),1.35*(200+90)+900)\n        state.update(epoch=11,phase=\'train\',batch_cursor=0)\n        self.assertEqual(d.current_epoch_projection(state,.2,200),900)\n\n    def test_hybrid_collation_uses_ordinary_path_on_both_tail_ranks(self):\n        events=np.arange(59)%27\n        for rank,size in ((0,16),(1,11)):\n            with patch.object(base,\'FeatureReader\',Reader):\n                ds=d.HybridCaptionBatches(\'unused\',events,rank=rank)\n                patches,_,ids=ds[0]\n                self.assertEqual(len(patches),len(set(ds.reader.data[\'caption_image\'][ids])))\n                self.assertLess(len(patches),len(ids))\n                tail=ds[1]\n                resumed=d.HybridCaptionBatches(\'unused\',events,rank=rank,cursor=1)[0]\n                self.assertEqual(len(tail[0]),size)\n                for actual,expected in zip(tail,resumed):\n                    np.testing.assert_array_equal(actual,expected)\n                np.testing.assert_array_equal(tail[0][:,0,0],ds.reader.data[\'caption_image\'][tail[2]])\n\n    def test_runner_static_accelerate_per_caption_resume(self):\n        source=(HERE/\'run_csa_full.py\').read_text(encoding=\'utf-8\');tree=ast.parse(source)\n        for value in (\'accelerator.prepare_model(model)\',\'accelerator.prepare_optimizer\',\'accelerator.backward(loss)\',\n                      \'context.events(data,events,device)\',\'current_epoch_projection(state,measured,val_est)\',\n                      "optimizer.load_state_dict(ck[\'optimizer\'])","rng_by_rank",\'mean_words\',\'short_le2\'):\n            self.assertIn(value,source)\n        self.assertNotIn(\'image_plan\',source)\n        self.assertNotIn(\'DDP(\',source);self.assertNotIn(\'remaining=(10*STEPS\',source)\n        call=next(n for n in ast.walk(tree) if isinstance(n,ast.Call) and isinstance(n.func,ast.Name) and n.func.id==\'Accelerator\')\n        self.assertEqual(next(k.value for k in call.keywords if k.arg==\'mixed_precision\').id,\'precision\')\n        ast.parse((HERE/\'csa_full_runtime.py\').read_text(encoding=\'utf-8\'))\n        ast.parse((HERE/\'score_csa_full.py\').read_text(encoding=\'utf-8\'))\n\n    def test_batch64_has_32_caption_events_per_rank(self):\n        events=np.arange(128)%27\n        for rank in (0,1):\n            with patch.object(base,\'FeatureReader\',Reader):\n                ds=d.SharedCaptionBatches(\'unused\',events,rank=rank,cursor=1,batch_size=64)\n                patches,seq,ids=ds[0]\n                expected=events[64+32*rank:96+32*rank]\n                np.testing.assert_array_equal(ids,expected)\n                np.testing.assert_array_equal(seq[:,1],ds.reader.data[\'sequences\'][expected,1])\n                self.assertEqual(len(patches),len(set(ds.reader.data[\'caption_image\'][expected])))\n\n\nif __name__==\'__main__\':main(verbosity=2)\n'}


In [ ]:
def run(args, timeout=1800):
    if time.monotonic()>=PARENT_DEADLINE:raise TimeoutError('Parent cap exceeded')
    timeout=min(timeout,PARENT_DEADLINE-time.monotonic())
    print('Running:', ' '.join(map(str, args)), flush=True)
    process = subprocess.Popen(list(map(str, args)), cwd=SOURCE, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, errors='replace', bufsize=1, start_new_session=True)
    messages = queue.Queue()
    def reader():
        try:
            for line in process.stdout: messages.put(line)
        finally: messages.put(None)
    thread = threading.Thread(target=reader, daemon=True); thread.start()
    started = heartbeat = time.monotonic(); ended = False
    with LOG.open('a', encoding='utf-8') as log:
        try:
            while not ended or process.poll() is None or not messages.empty():
                try:
                    line = messages.get(timeout=1)
                    if line is None: ended = True
                    else: print(line, end='', flush=True); log.write(line); log.flush()
                except queue.Empty: pass
                now = time.monotonic()
                if now-heartbeat >= 30:
                    print('Hybrid XE pilot active;', round(now-started), 'seconds', flush=True); heartbeat=now
                if now-started > timeout: raise TimeoutError('Phase/parent cap exceeded')
            if process.wait(): raise subprocess.CalledProcessError(process.returncode, args)
        finally:
            if process.poll() is None:
                os.killpg(process.pid, signal.SIGTERM)
                try: process.wait(timeout=10)
                except subprocess.TimeoutExpired: os.killpg(process.pid, signal.SIGKILL); process.wait()
            thread.join(timeout=5)


In [ ]:
complete=False
try:
    phase_times = {}
    setup_start = time.monotonic()
    sources = {name: HASHES[name] for name, destination in MAPPING.items() if destination not in PILOT_SOURCES}
    for name, digest in sources.items():
        download('https://raw.githubusercontent.com/Supzxjee/Image_Captioning/'+REVISION+'/'+name, digest, SOURCE/MAPPING[name])
    for name, source in PILOT_SOURCES.items():
        (SOURCE/name).write_bytes(source.encode('utf-8'))
    embedded_sha = {name: hashlib.sha256((SOURCE/name).read_bytes()).hexdigest() for name in PILOT_SOURCES}
    (OUTPUT/'source_receipt.json').write_text(json.dumps(dict(commit=REVISION, sha256=sources, embedded_sha256=embedded_sha), indent=2))
    run([sys.executable,'-u',SOURCE/'test_full_data.py'],120)
    run([sys.executable,'-u',SOURCE/'test_local_stage.py'],120)
    run([sys.executable,'-u',SOURCE/'test_semantic_data.py'],120)
    run([sys.executable,'-u',SOURCE/'test_csa_full_data.py'],120)
    run([sys.executable,'-u',SOURCE/'test_gradient_agreement.py'],120)
    run([sys.executable,'-u',SOURCE/'cuda_runtime_guard.py','--output',OUTPUT],60)

    run([sys.executable,'-m','pip','download','-q','--no-deps','--only-binary=:all:','--dest',SOURCE,'accelerate==1.15.0'],180)
    wheel=SOURCE/'accelerate-1.15.0-py3-none-any.whl'
    if hashlib.sha256(wheel.read_bytes()).hexdigest()!='97eacca0b73e45cb867dbf8c5d5d4dc32219544300e0c8992c7334dc2ef33cec':raise ValueError('Changed Accelerate wheel')
    run([sys.executable,'-m','pip','install','-q','--no-deps','--target',ACCEL_DEPS,wheel],180)
    run([sys.executable,'-c',"import accelerate;assert accelerate.__version__=='1.15.0';print('PinnedAccelerate',accelerate.__version__,flush=True)"],120)
    run([sys.executable,'-c',"from csa_patch_model import CSAPatchCaptioner;from csa_full_runtime import make_model;print('Neural module imports PASS; no model instantiated',flush=True)"],120)
    (OUTPUT/'accelerate_install.json').write_text(json.dumps(dict(version='1.15.0',wheel_sha256=hashlib.sha256(wheel.read_bytes()).hexdigest(),dependencies_reinstalled=False)))

    run([sys.executable,'-m','pip','install','-q','--no-deps','--target',DEPS,'pycocoevalcap==1.2'],600)
    os.environ['PYTHONPATH']=str(ACCEL_DEPS)+os.pathsep+str(DEPS)+os.pathsep+str(SOURCE)
    java_archive=SOURCE/'java8.tar.gz'
    download('https://github.com/adoptium/temurin8-binaries/releases/download/jdk8u472-b08/OpenJDK8U-jre_x64_linux_hotspot_8u472b08.tar.gz','6f7fb5fd640a0fd00837344b0920cbc4b9b9284b50e66f33789e3b250446a16e',java_archive)
    with tarfile.open(java_archive) as z:z.extractall(SOURCE/'jre',filter='data')
    java=list((SOURCE/'jre').rglob('bin/java'))
    if len(java)!=1:raise ValueError('Missing pinned Java8')
    os.environ['PATH']=str(java[0].parent)+os.pathsep+os.environ['PATH']
    run([java[0],'-version'],60)
    phase_times['setup_seconds'] = time.monotonic()-setup_start
    prep_start = time.monotonic()
    run([sys.executable,'-u',SOURCE/'run_csa_full.py','prepare','--config',CONFIG],1800)
    phase_times['prepare_seconds'] = time.monotonic()-prep_start
    launch=[sys.executable,'-u','-m','torch.distributed.run','--standalone','--nnodes=1','--nproc_per_node=2','--max_restarts=0']
    cfg['worker_seconds']=min(1200,PARENT_DEADLINE-time.monotonic()-1800);write_config()
    if cfg['worker_seconds']<180:raise TimeoutError('Insufficient budget for real IO preflight')
    preflight_start = time.monotonic()
    run([*launch,SOURCE/'run_csa_full.py','preflight','--config',CONFIG],cfg['worker_seconds']+30)
    phase_times['preflight_seconds'] = time.monotonic()-preflight_start
    (OUTPUT/'phase_timing.json').write_text(json.dumps(phase_times, indent=2))
    cfg['worker_seconds']=PARENT_DEADLINE-time.monotonic()-120;write_config()
    if cfg['worker_seconds']<1800:raise TimeoutError('Insufficient full-run budget after setup')
    run([*launch,SOURCE/'run_csa_full.py','worker','--config',CONFIG],cfg['worker_seconds']+30)
    state = json.loads((OUTPUT/'training_complete.json').read_text())
    if state['status']!='CSA_HYBRID_EPOCH1_COMPLETE' or state['updates']!=17711 or state['caption_events']!=566747 or len(state['history'])!=1:
        raise ValueError('Incomplete one-epoch accounting')
    result = state['history'][0]
    report = {k:result[k] for k in ('CIDEr','NLL','CIDEr_scoring_seconds','validation_wall_seconds')}
    train = []
    for rank in (0,1):
        rows = [json.loads(line) for line in (OUTPUT/f'train_profile_rank{rank}.jsonl').read_text().splitlines()]
        if [row['end_update'] for row in rows]!=[*range(100,17701,100),17711]:
            raise ValueError('Incomplete timing windows')
        samples = sum(row['sampled_steps'] for row in rows)
        if samples!=17711//20:
            raise ValueError('Incomplete CUDA event timing')
        valid = json.loads((OUTPUT/'evaluations'/'epoch_01'/f'worker{rank}.json').read_text())
        full = json.loads((OUTPUT/f'train_timing_rank{rank}.json').read_text())
        if valid['images']!=2500 or full['updates']!=17711:
            raise ValueError('Incomplete rank exposure')
        if full['precision']!='fp16' or full['optimizer_steps']+full['skipped_steps']!=17711 or full['nonfinite_norms']>full['skipped_steps']:
            raise ValueError('Inconsistent FP16 optimizer accounting')
        train.append(dict(rank=rank,train_seconds=full['seconds'],optimizer_steps=full['optimizer_steps'],
            skipped_steps=full['skipped_steps'],nonfinite_norms=full['nonfinite_norms'],
            peak_allocated_GiB=full['peak_allocated_GiB'],
            batch_wait_seconds=sum(row['batch_wait_seconds'] for row in rows),
            patch_token_copy_ms_per_sample=sum(row['patch_token_copy_ms'] for row in rows)/samples,
            training_step_ms_per_sample=sum(row['training_step_ms'] for row in rows)/samples,
            VALID_decode_seconds=valid['decode_seconds'],VALID_NLL_seconds=valid['NLL_seconds']))
    if train[0]['skipped_steps']!=train[1]['skipped_steps']:
        raise ValueError('Optimizer skips disagree by rank')
    report['ranks'] = train
    reference = {'train_seconds': 3245.148511599, 'CIDEr': 0.9619179304967791, 'NLL': 2.692114274493016}
    train_seconds = max(r['train_seconds'] for r in train)
    report['historical_fp32_reference'] = reference
    report['comparison'] = dict(fp16_train_seconds=train_seconds,
        fp16_over_fp32_throughput=reference['train_seconds']/train_seconds,
        CIDEr_delta_x100=100*(report['CIDEr']-reference['CIDEr']),NLL_delta=report['NLL']-reference['NLL'])
    (OUTPUT/'timing_summary.json').write_text(json.dumps(report,indent=2))
    complete = True
    print('TIMING SUMMARY',report,flush=True)
except BaseException:
    (OUTPUT/'failure.txt').write_text(traceback.format_exc());raise
finally:
    (OUTPUT/'run_status.json').write_text(json.dumps(dict(complete=complete,epochs_requested=1,training_precision="fp16",validation_precision="fp32",test_used=False,SCST=False,reranking=False)))
    review=OUTPUT.with_name(OUTPUT.name+'_review.zip')
    with zipfile.ZipFile(review,'w',zipfile.ZIP_DEFLATED) as z:
        for p in OUTPUT.rglob('*'):
            if p.is_file() and p.suffix in ('.json','.jsonl','.log','.txt'):z.write(p,str(p.relative_to(OUTPUT)))
    print('SEND REVIEW:',review,flush=True);print('KEEP ENTIRE OUTPUT:',OUTPUT,flush=True)
    print('LATEST:',OUTPUT/'checkpoint.bin','BEST VALID:',OUTPUT/'best.bin',flush=True)
    print('HYBRID XE EPOCH1 COMPLETE; VALID5000 scored.' if complete else 'INCOMPLETE; preserve weights/logs; no automatic restart or partial-model ranking.',flush=True)
